In [1]:
# %pip uninstall optuna xgboost lightgbm imbalanced-learn matplotlib seaborn scikit-learn pandas numpy
%pip install optuna xgboost lightgbm imbalanced-learn matplotlib seaborn scikit-learn pandas numpy

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


  Using cached optuna-4.4.0-py3-none-any.whl (395 kB)
  Using cached xgboost-2.1.4-py3-none-win_amd64.whl (124.9 MB)
  Using cached lightgbm-4.6.0-py3-none-win_amd64.whl (1.5 MB)
  Using cached imbalanced_learn-0.12.4-py3-none-any.whl (258 kB)
  Using cached colorlog-6.9.0-py3-none-any.whl (11 kB)
  Using cached alembic-1.16.2-py3-none-any.whl (242 kB)
  Using cached joblib-1.5.1-py3-none-any.whl (307 kB)
  Using cached mako-1.3.10-py3-none-any.whl (78 kB)


In [3]:
import pandas as pd

In [ ]:
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

train_df = train_df.replace([np.inf, -np.inf], np.nan).dropna()
test_df = test_df.replace([np.inf, -np.inf], np.nan).dropna()

# Now split into features and labels
y_train = train_df['label'].values
y_test = test_df['label'].values
X_train = train_df.drop('label', axis=1).values
X_test = test_df.drop('label', axis=1).values  # Fixed typo: was .value


In [14]:
# Required installs (if not installed already):
# pip install optuna xgboost lightgbm imbalanced-learn matplotlib seaborn scikit-learn

import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from imblearn.over_sampling import SMOTE
import optuna
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBClassifier
import lightgbm as lgb

# === 1. Prepare your dataset ===
# Assuming you already have:
# X_train, X_test, y_train, y_test from a previous split
# If not, uncomment and modify the following line:
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# === 2. Apply SMOTE to balance training data ===
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

# Feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_res)
X_test_scaled = scaler.transform(X_test)

# === 3. Optuna objective for both XGB and LGBM ===
def objective(trial, model_type='xgb'):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.3, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'random_state': 42
    }

    if model_type == 'lgb':
        params.update({'objective': 'binary', 'metric': 'auc', 'verbosity': -1})
        model = lgb.LGBMClassifier(**params)
        model.fit(X_train_scaled, y_train_res,
                  eval_set=[(X_test_scaled, y_test)],
                )
    else:
        params.update({'objective': 'binary:logistic', 'eval_metric': 'auc', 'verbosity': 0})
        model = XGBClassifier(**params)
        model.fit(X_train_scaled, y_train_res,
                  eval_set=[(X_test_scaled, y_test)],
                  )

    proba = model.predict_proba(X_test_scaled)[:, 1]
    return roc_auc_score(y_test, proba)

# === 4. Run Optuna studies ===
sampler = optuna.samplers.RandomSampler(seed=42)

study_xgb = optuna.create_study(direction='maximize', sampler=sampler)
study_xgb.optimize(lambda t: objective(t, 'xgb'), n_trials=30)
print("XGB best AUC:", study_xgb.best_value, "params:", study_xgb.best_params)

study_lgb = optuna.create_study(direction='maximize', sampler=sampler)
study_lgb.optimize(lambda t: objective(t, 'lgb'), n_trials=30)
print("LGBM best AUC:", study_lgb.best_value, "params:", study_lgb.best_params)

# === 5. Train final models and evaluate ===
def train_and_eval(model_cls, best_params):
    best_params = best_params.copy()
    best_params.update({'random_state': 42})

    model = model_cls(**best_params)
    model.fit(X_train_scaled, y_train_res)

    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]
    cm = confusion_matrix(y_test, y_pred)
    report = classification_report(y_test, y_pred, digits=4, output_dict=True)
    auc = roc_auc_score(y_test, y_proba)

    # Plot confusion matrix
    plt.figure(figsize=(5,5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f"{model_cls.__name__} Confusion Matrix")
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()

    print(f"=== {model_cls.__name__} Metrics ===")
    print(f"Precision: {report['1']['precision']:.4f}")
    print(f"Recall:    {report['1']['recall']:.4f}")
    print(f"F1-score:  {report['1']['f1-score']:.4f}")
    print(f"AUROC:     {auc:.4f}\n")

# Final training and evaluation
xgb_params = study_xgb.best_params.copy()
xgb_params.update({'objective': 'binary:logistic', 'eval_metric': 'auc', 'verbosity': 0})
train_and_eval(XGBClassifier, xgb_params)

lgb_params = study_lgb.best_params.copy()
lgb_params.update({'objective': 'binary', 'metric': 'auc', 'verbosity': -1})
train_and_eval(lgb.LGBMClassifier, lgb_params)


ValueError: Input contains NaN, infinity or a value too large for dtype('float64').

In [12]:
%pip install tensorflow

Defaulting to user installation because normal site-packages is not writeable
  Using cached numpy-2.0.2-cp39-cp39-win_amd64.whl (15.9 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.22.4
    Uninstalling numpy-1.22.4:
      Successfully uninstalled numpy-1.22.4
Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'C:\\Users\\Admin\\AppData\\Roaming\\Python\\Python39\\site-packages\\~-mpy\\.libs\\libopenblas.EL2C6PLE4ZYW3ECEVIV3OXXGRN2NRFM2.gfortran-win_amd64.dll'
Check the permissions.



In [4]:
# === INSTALL REQUIRED LIBRARIES ===
# pip install pandas numpy scikit-learn imbalanced-learn xgboost lightgbm optuna torch

import pandas as pd
import numpy as np
import optuna
# import matplotlib.pyplot as plt
# import seaborn as sns

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

from xgboost import XGBClassifier
import lightgbm as lgb

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# === 1. LOAD & CLEAN DATA ===
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

train_df = train_df.replace([np.inf, -np.inf], np.nan).dropna()
test_df = test_df.replace([np.inf, -np.inf], np.nan).dropna()

y_train = train_df['label'].values
y_test = test_df['label'].values
X_train = train_df.drop('label', axis=1).values
X_test = test_df.drop('label', axis=1).values

# === 2. SMOTE & SCALING ===
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_res)
X_test_scaled = scaler.transform(X_test)

# === 3. TUNE BASE MODELS WITH OPTUNA ===
def objective(trial, model_type='xgb'):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 300),
        'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'random_state': 42
    }

    if model_type == 'xgb':
        params.update({'objective': 'binary:logistic', 'eval_metric': 'auc', 'verbosity': 0})
        model = XGBClassifier(**params)
    else:
        params.update({'objective': 'binary', 'metric': 'auc', 'verbosity': -1})
        model = lgb.LGBMClassifier(**params)

    model.fit(X_train_scaled, y_train_res)
    preds = model.predict_proba(X_test_scaled)[:, 1]
    return roc_auc_score(y_test, preds)

xgb_study = optuna.create_study(direction='maximize')
xgb_study.optimize(lambda t: objective(t, 'xgb'), n_trials=20)

lgb_study = optuna.create_study(direction='maximize')
lgb_study.optimize(lambda t: objective(t, 'lgb'), n_trials=20)

xgb_params = xgb_study.best_params
xgb_params.update({'objective': 'binary:logistic', 'eval_metric': 'auc', 'verbosity': 0})

lgb_params = lgb_study.best_params
lgb_params.update({'objective': 'binary', 'metric': 'auc', 'verbosity': -1})

# === 4. K-FOLD STACKING TO CREATE META-FEATURES ===
n_folds = 5
skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)

oof_train = np.zeros((X_train_scaled.shape[0], 2))
test_meta = np.zeros((X_test_scaled.shape[0], 2, n_folds))

for i, (train_idx, val_idx) in enumerate(skf.split(X_train_scaled, y_train_res)):
    X_tr, X_val = X_train_scaled[train_idx], X_train_scaled[val_idx]
    y_tr, y_val = y_train_res[train_idx], y_train_res[val_idx]

    xgb = XGBClassifier(**xgb_params)
    lgbm = lgb.LGBMClassifier(**lgb_params)

    xgb.fit(X_tr, y_tr)
    lgbm.fit(X_tr, y_tr)

    oof_train[val_idx, 0] = xgb.predict_proba(X_val)[:, 1]
    oof_train[val_idx, 1] = lgbm.predict_proba(X_val)[:, 1]

    test_meta[:, 0, i] = xgb.predict_proba(X_test_scaled)[:, 1]
    test_meta[:, 1, i] = lgbm.predict_proba(X_test_scaled)[:, 1]

X_meta_train = oof_train
X_meta_test = test_meta.mean(axis=2)

# Convert meta features and labels to torch tensors
X_meta_train_t = torch.tensor(X_meta_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train_res, dtype=torch.float32).unsqueeze(1)
X_meta_test_t = torch.tensor(X_meta_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

# === 5. DEFINE PYTORCH META-NN ===
class MetaNN(nn.Module):
    def __init__(self, input_dim, n_layers, hidden_units, dropout):
        super(MetaNN, self).__init__()
        layers = []
        current_dim = input_dim
        for i in range(n_layers):
            layers.append(nn.Linear(current_dim, hidden_units[i]))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            current_dim = hidden_units[i]
        layers.append(nn.Linear(current_dim, 1))
        layers.append(nn.Sigmoid())
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

# Training function for PyTorch model
def train_pytorch_model(model, optimizer, criterion, train_loader, val_loader, epochs, device):
    model.to(device)
    best_val_auc = 0
    best_state = None

    for epoch in range(epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            preds = model(xb)
            loss = criterion(preds, yb)
            loss.backward()
            optimizer.step()

        # Validation AUC
        model.eval()
        val_preds = []
        val_labels = []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                preds = model(xb)
                val_preds.append(preds.cpu())
                val_labels.append(yb.cpu())

        val_preds = torch.cat(val_preds).numpy()
        val_labels = torch.cat(val_labels).numpy()
        val_auc = roc_auc_score(val_labels, val_preds)
        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_state = model.state_dict()

    model.load_state_dict(best_state)
    return best_val_auc

# === 6. OPTUNA OBJECTIVE FOR PYTORCH META-NN ===
def pytorch_nn_objective(trial):
    n_layers = trial.suggest_int("n_layers", 1, 4)
    dropout = trial.suggest_float("dropout", 0.1, 0.5)
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    batch_size = trial.suggest_categorical("batch_size", [32, 64])
    epochs = trial.suggest_int("epochs", 10, 30)

    hidden_units = [trial.suggest_int(f"units_l{i}", 8, 80) for i in range(n_layers)]

    dataset = TensorDataset(X_meta_train_t, y_train_t)
    train_size = int(0.8 * len(dataset))
    val_size = len(dataset) - train_size
    train_ds, val_ds = torch.utils.data.random_split(dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42))

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = MetaNN(input_dim=X_meta_train.shape[1], n_layers=n_layers, hidden_units=hidden_units, dropout=dropout)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCELoss()

    val_auc = train_pytorch_model(model, optimizer, criterion, train_loader, val_loader, epochs, device)
    return val_auc

study = optuna.create_study(direction='maximize')
study.optimize(pytorch_nn_objective, n_trials=50)

print("Best PyTorch Meta-NN Params:", study.best_params)
print("Best Validation AUC:", study.best_value)

# === 7. TRAIN FINAL META-NN MODEL WITH BEST PARAMS ===
best = study.best_params
n_layers = best.pop('n_layers')
epochs = best.pop('epochs')
batch_size = best.pop('batch_size')
dropout = best.pop('dropout')
lr = best.pop('lr')
hidden_units = [best[f'units_l{i}'] for i in range(n_layers)]

final_model = MetaNN(input_dim=X_meta_train.shape[1], n_layers=n_layers, hidden_units=hidden_units, dropout=dropout)
optimizer = optim.Adam(final_model.parameters(), lr=lr)
criterion = nn.BCELoss()

full_dataset = TensorDataset(X_meta_train_t, y_train_t)
full_loader = DataLoader(full_dataset, batch_size=batch_size, shuffle=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
final_model.to(device)

final_model.train()
for epoch in range(epochs):
    for xb, yb in full_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        preds = final_model(xb)
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()

# === 8. EVALUATE FINAL STACKED MODEL ===
final_model.eval()
with torch.no_grad():
    preds_test = final_model(X_meta_test_t.to(device)).cpu().numpy().ravel()

preds_class = (preds_test > 0.5).astype(int)

cm = confusion_matrix(y_test, preds_class)
report = classification_report(y_test, preds_class, digits=4)
auc = roc_auc_score(y_test, preds_test)

# Commented out plotting
# plt.figure(figsize=(5,5))
# sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
# plt.title("Stacked Meta-Model (PyTorch NN) Confusion Matrix")
# plt.xlabel("Predicted")
# plt.ylabel("Actual")
# plt.show()

print("=== Final PyTorch Meta-Model Performance ===")
print(report)
print(f"AUROC: {auc:.4f}")




[I 2025-06-26 14:47:53,986] A new study created in memory with name: no-name-9b72227a-4af5-40c1-92ae-e19106fdbb92
[I 2025-06-26 14:48:00,672] Trial 0 finished with value: 0.5037194294909058 and parameters: {'n_estimators': 129, 'learning_rate': 0.0175834520832845, 'max_depth': 8, 'subsample': 0.8597510203250416, 'colsample_bytree': 0.8035107449610135}. Best is trial 0 with value: 0.5037194294909058.
[I 2025-06-26 14:48:02,752] Trial 1 finished with value: 0.5296897246021517 and parameters: {'n_estimators': 167, 'learning_rate': 0.011371377246321316, 'max_depth': 5, 'subsample': 0.6941497195034022, 'colsample_bytree': 0.699562243090159}. Best is trial 1 with value: 0.5296897246021517.
[I 2025-06-26 14:48:06,630] Trial 2 finished with value: 0.4946226232297959 and parameters: {'n_estimators': 128, 'learning_rate': 0.0017273509933066293, 'max_depth': 9, 'subsample': 0.7014213753335832, 'colsample_bytree': 0.6819421139135444}. Best is trial 1 with value: 0.5296897246021517.
[I 2025-06-26 1

Best PyTorch Meta-NN Params: {'n_layers': 3, 'dropout': 0.38849989881436, 'lr': 0.0004229434580635613, 'batch_size': 64, 'epochs': 11, 'units_l0': 65, 'units_l1': 40, 'units_l2': 9}
Best Validation AUC: 0.7220944696587661
=== Final PyTorch Meta-Model Performance ===
              precision    recall  f1-score   support

           0     0.8473    1.0000    0.9174      1815
           1     0.0000    0.0000    0.0000       327

    accuracy                         0.8473      2142
   macro avg     0.4237    0.5000    0.4587      2142
weighted avg     0.7180    0.8473    0.7773      2142

AUROC: 0.5742


c:\bt23ece064\training\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\bt23ece064\training\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\bt23ece064\training\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [3]:
print(cm)

[[1550  265]
 [ 262   65]]
